### Hybrid Oncology Pharmacovigilance System  


### Objective
This notebook collects large-scale adverse event data from the FDA FAERS database using the openFDA API.

We use pagination to fetch 3000+ records for oncology drugs.

Data will be stored in:

- `data/raw/faers/`
- `data/processed/`


In [13]:
import requests
import json
import pandas as pd
import os
import time


### Configuration

We define:
- Drug name
- Total records to fetch
- Batch size (API allows max 100 per request)


In [14]:
DRUG_NAME = "DOXORUBICIN"
TOTAL_RECORDS = 3000
BATCH_SIZE = 100
BASE_URL = "https://api.fda.gov/drug/event.json"

print("Configuration Loaded.")


Configuration Loaded.


### Create Project Folder Structure

We ensure required folders exist before saving data.


In [17]:
os.makedirs("../data/raw/faers", exist_ok=True)
os.makedirs("../data/processed", exist_ok=True)

print("Folders created successfully.")


Folders created successfully.


### Pagination Function for Large Data Fetching

The openFDA API allows:
- Maximum 100 records per request

We use:
- `limit=100`
- `skip` parameter for pagination


In [18]:
def fetch_large_faers(drug_name, total_records, batch_size):
    
    all_results = []
    
    for skip in range(0, total_records, batch_size):
        
        params = {
            "search": f'patient.drug.medicinalproduct:"{drug_name}"',
            "limit": batch_size,
            "skip": skip
        }
        
        print(f"Fetching records {skip} to {skip + batch_size}...")
        
        response = requests.get(BASE_URL, params=params)
        
        if response.status_code == 200:
            data = response.json()
            results = data.get("results", [])
            
            if not results:
                print("No more records available.")
                break
            
            all_results.extend(results)
            
            # Avoid API rate limit
            time.sleep(0.3)
        
        else:
            print("Error:", response.status_code)
            break
    
    print(f"\nTotal records fetched: {len(all_results)}")
    return all_results


In [19]:
faers_large_data = fetch_large_faers(DRUG_NAME, TOTAL_RECORDS, BATCH_SIZE)


Fetching records 0 to 100...
Fetching records 100 to 200...
Fetching records 200 to 300...
Fetching records 300 to 400...
Fetching records 400 to 500...
Fetching records 500 to 600...
Fetching records 600 to 700...
Fetching records 700 to 800...
Fetching records 800 to 900...
Fetching records 900 to 1000...
Fetching records 1000 to 1100...
Fetching records 1100 to 1200...
Fetching records 1200 to 1300...
Fetching records 1300 to 1400...
Fetching records 1400 to 1500...
Fetching records 1500 to 1600...
Fetching records 1600 to 1700...
Fetching records 1700 to 1800...
Fetching records 1800 to 1900...
Fetching records 1900 to 2000...
Fetching records 2000 to 2100...
Fetching records 2100 to 2200...
Fetching records 2200 to 2300...
Fetching records 2300 to 2400...
Fetching records 2400 to 2500...
Fetching records 2500 to 2600...
Fetching records 2600 to 2700...
Fetching records 2700 to 2800...
Fetching records 2800 to 2900...
Fetching records 2900 to 3000...

Total records fetched: 3000


### Save Raw JSON Data

In [21]:
raw_path = f"../data/raw/faers/{DRUG_NAME.lower()}_3000_raw.json"

with open(raw_path, "w") as f:
    json.dump(faers_large_data, f)

print("Raw JSON saved at:", raw_path)


Raw JSON saved at: ../data/raw/faers/doxorubicin_3000_raw.json


### Convert JSON to Structured DataFrame

We extract:
- Drug
- Reaction
- Serious
- Age
- Sex
- Report_Date


In [22]:
records = []

for report in faers_large_data:
    
    try:
        reaction = report["patient"]["reaction"][0]["reactionmeddrapt"]
    except:
        reaction = None
    
    records.append({
        "Drug": DRUG_NAME,
        "Reaction": reaction,
        "Serious": report.get("serious"),
        "Age": report["patient"].get("patientonsetage"),
        "Sex": report["patient"].get("patientsex"),
        "Report_Date": report.get("receivedate")
    })

df_large = pd.DataFrame(records)

print("DataFrame Created.")
df_large.head()


DataFrame Created.


,Drug,Reaction,Serious,Age,Sex,Report_Date
0,DOXORUBICIN,Hepatocellular injury,1,16,2,20140312
1,DOXORUBICIN,Malignant neoplasm progression,1,60,1,20140312
2,DOXORUBICIN,Septic shock,1,69,2,20140312
3,DOXORUBICIN,Hyperkalaemia,1,67,1,20140312
4,DOXORUBICIN,Leukoencephalopathy,1,69,2,20140313


### Save Processed CSV

In [23]:
csv_path = f"../data/processed/{DRUG_NAME.lower()}_3000_faers.csv"

df_large.to_csv(csv_path, index=False)

print("Processed CSV saved at:", csv_path)
print("Final Shape:", df_large.shape)


Processed CSV saved at: ../data/processed/doxorubicin_3000_faers.csv
Final Shape: (3000, 6)


# Data Collection Completed Successfully

I have now collected 3000+ FAERS reports.

Next Steps:
- Perform large-scale EDA
- Add multiple oncology drugs
- Perform PRR signal detection
- Build hybrid risk model
